In [10]:
import torch

In [11]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

In [12]:
from torchvision.datasets import CIFAR10

train_set = CIFAR10(root="./data", train=True, download=True, transform=transform)
test_set = CIFAR10(root="./data", train=False, download=True, transform=transform)

In [13]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
test_loader = DataLoader(test_set, batch_size=128, shuffle=True)

In [14]:
import torch.nn as nn

In [21]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 2 * 2, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.conv(x)
        return self.network(x)

In [22]:
learning_rate = 1e-3
epochs = 20
batch_size = 128

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Model()
model.to(device)
print(model)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

Model(
  (conv): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (14

In [23]:
def train(model, train_loader, optimizer, loss_fn, device):
    model.train()
    for batch, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = loss_fn(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch % 100 == 0:
            print(f"Loss: {loss.item()}")


def test(model, test_loader, loss_fn, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = loss_fn(outputs, labels)
            total_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_loss = total_loss / len(test_loader)
    accuracy = correct / total
    print(f"Test Loss: {avg_loss:.4f}, Accuracy: {accuracy*100:.4f}")

In [24]:
for epoch in range(epochs):
    print(f"Epoch {epoch+1}/{epochs} --------------------")
    train(model, train_loader, optimizer, loss_fn, device)
    test(model, test_loader, loss_fn, device)

Epoch 1/20 --------------------
Loss: 2.324139356613159
Loss: 1.7557811737060547
Loss: 1.5720343589782715
Loss: 1.5047221183776855
Test Loss: 1.2760, Accuracy: 53.0300
Epoch 2/20 --------------------
Loss: 1.0938143730163574
Loss: 1.3438572883605957
Loss: 1.258333444595337
Loss: 1.1238019466400146
Test Loss: 1.0717, Accuracy: 61.0300
Epoch 3/20 --------------------
Loss: 1.2607795000076294
Loss: 1.0451281070709229
Loss: 0.9932594299316406
Loss: 1.0050112009048462
Test Loss: 1.0594, Accuracy: 62.3300
Epoch 4/20 --------------------
Loss: 0.9028061032295227
Loss: 1.1030023097991943
Loss: 1.2477660179138184
Loss: 0.9797064065933228
Test Loss: 0.8749, Accuracy: 69.9700
Epoch 5/20 --------------------
Loss: 0.7549707889556885
Loss: 0.9683231115341187
Loss: 0.7561911940574646
Loss: 0.6878958940505981
Test Loss: 0.9444, Accuracy: 66.7500
Epoch 6/20 --------------------
Loss: 0.773152232170105
Loss: 0.8631373643875122
Loss: 0.7842512130737305
Loss: 0.7729959487915039
Test Loss: 0.8535, Accurac